<a href="https://colab.research.google.com/github/HopeSilkina/deposits_forecast_project/blob/main/notebooks/01_EDA_Modeling_Deposits_Forecast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# FORECASTING HOUSEHOLD DEPOSITS IN RUSSIA
# Data Science Portfolio Project
# Author: Nadezhda Silkina
# Date: 2026
# ============================================================

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson
from statsmodels.tsa.stattools import adfuller
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# Plot settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ All libraries loaded successfully")
print(f"Pandas version: {pd.__version__}")

# ============================================================
# 2. LOAD DATA
# ============================================================

url = 'https://raw.githubusercontent.com/HopeSilkina/deposits_forecast_project/main/data/processed_deposits_data.xlsx'
df = pd.read_excel(url, sheet_name='data')

# Convert date column
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)

# Sort by date
df.sort_index(inplace=True)

print(f"✅ Data loaded. Records: {len(df)}")
print(f"Period: from {df.index.min()} to {df.index.max()}")
print("\nFirst 5 rows:")
print(df.head())

# ============================================================
# 3. EXPLORATORY DATA ANALYSIS (EDA)
# ============================================================

print("\n" + "="*60)
print("3. EXPLORATORY DATA ANALYSIS")
print("="*60)

# 3.1. Descriptive statistics
print("\n📊 Descriptive statistics:")
print(df.describe())

# 3.2. Missing values check
print("\n📊 Missing values:")
print(df.isnull().sum())

# 3.3. Correlation matrix (heatmap)
plt.figure(figsize=(12, 10))
corr_matrix = df.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix of All Features', fontsize=14)
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# 3.4. Time series visualization
fig, axes = plt.subplots(nrows=5, ncols=2, figsize=(14, 12))
axes = axes.flatten()

# Select only original features for time series
original_features = ['DEPOS', 'WAGE', 'SERV', 'DEP1', 'CRED1', 'CPI', 'USDind', 'UNEM', 'IPI', 'IMP']

for idx, col in enumerate(original_features):
    ax = axes[idx]
    ax.plot(df.index, df[col], linewidth=1.5)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('')
    ax.grid(True, alpha=0.3)

# Remove empty subplots (if any)
for idx in range(len(original_features), len(axes)):
    fig.delaxes(axes[idx])

plt.suptitle('Time Series of All Indicators', fontsize=14, y=0.98)
plt.tight_layout()
plt.savefig('time_series_all.png', dpi=300, bbox_inches='tight')
plt.show()

# 3.5. Scatter plots: DEPOS vs each original feature
print("\n📊 Building scatter plots: DEPOS vs each predictor...")

# Original features (excluding DEPOS)
predictors = ['WAGE', 'SERV', 'DEP1', 'CRED1', 'CPI', 'USDind', 'UNEM', 'IPI', 'IMP']
n_features = len(predictors)

# Calculate grid size (3x3 = 9)
n_cols = 3
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(nrows=n_rows, ncols=n_cols, figsize=(15, 5 * n_rows))
axes = axes.flatten()

for idx, col in enumerate(predictors):
    ax = axes[idx]
    ax.scatter(df[col], df['DEPOS'], alpha=0.6, s=30, color='steelblue', edgecolor='white')

    # Add trend line (linear regression)
    mask = ~(df[col].isna() | df['DEPOS'].isna())
    if mask.sum() > 1:
        z = np.polyfit(df.loc[mask, col], df.loc[mask, 'DEPOS'], 1)
        p = np.poly1d(z)
        x_sorted = np.sort(df.loc[mask, col])
        ax.plot(x_sorted, p(x_sorted), "r--", linewidth=1.5,
                label=f'R² = {np.corrcoef(df.loc[mask, col], df.loc[mask, "DEPOS"])[0,1]**2:.3f}')

    ax.set_xlabel(col, fontsize=10)
    ax.set_ylabel('DEPOS', fontsize=10)
    ax.set_title(f'DEPOS vs {col}', fontsize=11)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

# Remove empty subplots (if any)
for idx in range(n_features, len(axes)):
    fig.delaxes(axes[idx])

plt.suptitle('Scatter Plots: Deposit Volume vs Macroeconomic Indicators',
             fontsize=14, y=0.98)
plt.tight_layout()
plt.savefig('scatter_plots.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Scatter plots saved to 'scatter_plots.png'")

# 3.6. Target variable distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['DEPOS'], bins=20, edgecolor='black', alpha=0.7)
axes[0].set_title('Histogram: Deposit Volume (DEPOS)')
axes[0].set_xlabel('Billion RUB')
axes[0].set_ylabel('Frequency')

axes[1].boxplot(df['DEPOS'])
axes[1].set_title('Box Plot: Deposit Volume (DEPOS)')
axes[1].set_ylabel('Billion RUB')

plt.tight_layout()
plt.savefig('depos_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Exploratory data analysis completed")

# ============================================================
# 4. ECONOMETRIC DIAGNOSTICS
# ============================================================

print("\n" + "="*60)
print("4. ECONOMETRIC DIAGNOSTICS")
print("="*60)

# 4.1. Stationarity test (Augmented Dickey-Fuller)
print("\n🔍 Dickey-Fuller test for DEPOS:")

def adf_test(series, series_name):
    result = adfuller(series, autolag='AIC')
    print(f"\n  {series_name}:")
    print(f"    ADF statistic: {result[0]:.4f}")
    print(f"    p-value: {result[1]:.4f}")
    print(f"    Critical values:")
    for key, value in result[4].items():
        print(f"      {key}: {value:.4f}")
    print(f"    Conclusion: {'Stationary ✅' if result[1] < 0.05 else 'Non-stationary ❌'}")

adf_test(df['DEPOS'], 'DEPOS')

# 4.2. Multicollinearity (VIF)
print("\n🔍 Multicollinearity (VIF):")

X_vif = df.drop('DEPOS', axis=1)
# Add constant for VIF
X_vif_with_const = sm.add_constant(X_vif)

vif_data = pd.DataFrame()
vif_data['feature'] = X_vif.columns
vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]

print(vif_data.sort_values('VIF', ascending=False))
print("\n📌 Interpretation: VIF > 10 indicates strong multicollinearity")

# ============================================================
# 5. DATA PREPARATION FOR MODELING
# ============================================================

print("\n" + "="*60)
print("5. DATA PREPARATION FOR MODELING")
print("="*60)

# 5.1. Log transformation of target variable
df['DEPOS_log'] = np.log(df['DEPOS'])

# 5.2. Adding lag features
for lag in [1, 3, 6, 12]:
    df[f'DEPOS_lag_{lag}'] = df['DEPOS'].shift(lag)

# 5.3. Prepare features and target variable
X = df.drop(['DEPOS', 'DEPOS_log'], axis=1).dropna()
y = df.loc[X.index, 'DEPOS']

print(f"📊 X shape: {X.shape}")
print(f"📊 y shape: {y.shape}")

# 5.4. Train/test split (last 12 months for testing)
train_size = len(X) - 12
X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

print(f"\n📊 Training set: {len(X_train)} records")
print(f"📊 Test set: {len(X_test)} records")
print(f"📊 Test period: {X.index[train_size]} — {X.index[-1]}")

# 5.5. Feature scaling (for Ridge regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ============================================================
# 6. MODEL TRAINING AND COMPARISON
# ============================================================

print("\n" + "="*60)
print("6. MODEL TRAINING AND COMPARISON")
print("="*60)

models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)
}

results = {}

for name, model in models.items():
    print(f"\n🔧 Training {name}...")

    # Use scaled data for Ridge, original for others
    if name == 'Ridge Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    # Performance metrics
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

    results[name] = {
        'R²': r2,
        'MAE': mae,
        'RMSE': rmse,
        'MAPE': f"{mape:.2f}%",
        'predictions': y_pred,
        'model': model
    }

    print(f"  ✅ R²: {r2:.4f}")
    print(f"  ✅ MAE: {mae:.2f} bln RUB")
    print(f"  ✅ RMSE: {rmse:.2f} bln RUB")
    print(f"  ✅ MAPE: {mape:.2f}%")

# ============================================================
# 6.1. COMPARISON TABLE (TEST SET)
# ============================================================

results_df = pd.DataFrame({
    name: {k: v for k, v in res.items() if k not in ['predictions', 'model']}
    for name, res in results.items()
}).T

print("\n" + "="*60)
print("📊 MODEL COMPARISON TABLE (TEST SET):")
print("="*60)
print(results_df.round(4))

# ============================================================
# 6.2. FULL METRICS OF BEST MODEL (TRAINING SET)
# ============================================================

print("\n" + "="*60)
print("6.2. FULL METRICS OF BEST MODEL (TRAINING + TEST)")
print("="*60)

# Determine best model by test R²
best_model_name = results_df['R²'].idxmax()
best_model_results = results[best_model_name]
best_model = best_model_results['model']

# Predictions on training set
if 'Ridge' in best_model_name:
    y_train_pred_best = best_model.predict(X_train_scaled)
else:
    y_train_pred_best = best_model.predict(X_train)

# Predictions on test set
y_test_pred_best = best_model_results['predictions']

# Training set metrics
n_train = len(y_train)
n_params = X_train.shape[1] + 1  # +1 for intercept (all models have intercept)

r2_train_best = r2_score(y_train, y_train_pred_best)

# Adjusted R² on training set
if n_train - n_params - 1 > 0:
    r2_adj_train_best = 1 - (1 - r2_train_best) * (n_train - 1) / (n_train - n_params - 1)
else:
    r2_adj_train_best = np.nan

# AIC and BIC on training set
residuals_train_best = y_train.values - y_train_pred_best
rss_train_best = np.sum(residuals_train_best**2)

# Unbiased variance estimator (important for correct AIC/BIC)
sigma2_best = rss_train_best / (n_train - n_params)
log_likelihood_best = -0.5 * n_train * (np.log(2 * np.pi * sigma2_best) + 1)

aic_best = -2 * log_likelihood_best + 2 * n_params
bic_best = -2 * log_likelihood_best + n_params * np.log(n_train)

# Test set metrics (already calculated)
r2_test_best = best_model_results['R²']
rmse_test_best = best_model_results['RMSE']
mae_test_best = best_model_results['MAE']

print(f"\n📊 {best_model_name}:")
print(f"   TRAINING set (n={n_train}):")
print(f"     R²_train = {r2_train_best:.4f}")
print(f"     R²_adj_train = {r2_adj_train_best:.4f}")
print(f"     AIC = {aic_best:.2f}")
print(f"     BIC = {bic_best:.2f}")
print(f"     RSS = {rss_train_best:.2f}")
print(f"   TEST set (n={len(y_test)}):")
print(f"     R²_test = {r2_test_best:.4f}")
print(f"     RMSE = {rmse_test_best:.2f} bln RUB")
print(f"     MAE = {mae_test_best:.2f} bln RUB")

# Overfitting check
r2_diff = r2_train_best - r2_test_best
if r2_diff > 0.1:
    print(f"\n   ⚠️ R²_train - R²_test gap = {r2_diff:.4f} — possible overfitting")
elif r2_diff > 0.05:
    print(f"\n   ℹ️ R²_train - R²_test gap = {r2_diff:.4f} — moderate")
else:
    print(f"\n   ✅ R²_train - R²_test gap = {r2_diff:.4f} — model generalizes well")

# Save for use in subsequent blocks
best_model_metrics = {
    'name': best_model_name,
    'r2_train': r2_train_best,
    'r2_adj_train': r2_adj_train_best,
    'aic': aic_best,
    'bic': bic_best,
    'r2_test': r2_test_best,
    'rmse_test': rmse_test_best,
    'mae_test': mae_test_best,
    'n_params': n_params,
    'n_train': n_train,
    'y_train_pred': y_train_pred_best,
    'y_test_pred': y_test_pred_best
}

# ============================================================
# 6.3. RESIDUAL DIAGNOSTICS OF BEST MODEL (TRAINING SET)
# ============================================================

print("\n" + "="*60)
print("6.3. RESIDUAL DIAGNOSTICS (TRAINING SET)")
print("="*60)

from scipy.stats import shapiro
from statsmodels.stats.diagnostic import het_breuschpagan, acorr_breusch_godfrey
from statsmodels.graphics.tsaplots import plot_acf

# Residuals on training set
residuals_train = y_train.values - y_train_pred_best

# 6.3.1. Normality test (Shapiro-Wilk)
if 3 <= n_train <= 5000:  # Shapiro-Wilk valid for this sample size
    shapiro_stat, shapiro_p = shapiro(residuals_train)
    normality_ok = shapiro_p > 0.05
else:
    shapiro_stat, shapiro_p = np.nan, np.nan
    normality_ok = None

# 6.3.2. Homoscedasticity test (Breusch-Pagan)
try:
    exog_bp = sm.add_constant(y_train_pred_best.reshape(-1, 1))
    bp_stat, bp_p, bp_f, bp_f_p = het_breuschpagan(residuals_train, exog_bp)
    homoscedasticity_ok = bp_p > 0.05
except Exception as e:
    bp_stat, bp_p = np.nan, np.nan
    homoscedasticity_ok = None
    print(f"   ⚠️ Breusch-Pagan test error: {e}")

# 6.3.3. Autocorrelation test (Breusch-Godfrey) — replaces DW
# Advantages over DW:
# - Valid with lagged dependent variables
# - Tests multiple autocorrelation orders
try:
    exog_bg = sm.add_constant(y_train_pred_best.reshape(-1, 1))
    bg_model = sm.OLS(residuals_train, exog_bg).fit()

    # Lag-1 autocorrelation test
    bg_stat_1, bg_p_1, _, _ = acorr_breusch_godfrey(bg_model, nlags=1)
    autocorr_1_ok = bg_p_1 > 0.05

    # Lag-4 autocorrelation test (seasonal)
    bg_stat_4, bg_p_4, _, _ = acorr_breusch_godfrey(bg_model, nlags=4)
    autocorr_4_ok = bg_p_4 > 0.05
except Exception as e:
    bg_stat_1, bg_p_1 = np.nan, np.nan
    bg_stat_4, bg_p_4 = np.nan, np.nan
    autocorr_1_ok = None
    autocorr_4_ok = None
    print(f"   ⚠️ Breusch-Godfrey test error: {e}")

# Print diagnostic results
print(f"\n🔍 Diagnostic results for {best_model_name}:")
print(f"   Residual normality:")
print(f"     Shapiro-Wilk stat = {shapiro_stat:.4f}, p = {shapiro_p:.4f}")
print(f"     {'✅ Residuals are normal' if normality_ok else '⚠️ Deviation from normality' if normality_ok is not None else 'N/A'}")
print(f"   Homoscedasticity:")
print(f"     Breusch-Pagan stat = {bp_stat:.4f}, p = {bp_p:.4f}")
print(f"     {'✅ Homoscedasticity' if homoscedasticity_ok else '⚠️ Heteroscedasticity' if homoscedasticity_ok is not None else 'N/A'}")
print(f"   Autocorrelation (Breusch-Godfrey test):")
print(f"     Lag 1: stat = {bg_stat_1:.4f}, p = {bg_p_1:.4f} → {'✅ No autocorrelation' if autocorr_1_ok else '⚠️ Autocorrelation' if autocorr_1_ok is not None else 'N/A'}")
print(f"     Lag 4: stat = {bg_stat_4:.4f}, p = {bg_p_4:.4f} → {'✅ No autocorrelation' if autocorr_4_ok else '⚠️ Autocorrelation' if autocorr_4_ok is not None else 'N/A'}")

# Save diagnostic results
best_model_diagnostics = {
    'shapiro_p': shapiro_p,
    'normality': normality_ok,
    'bp_p': bp_p,
    'homoscedasticity': homoscedasticity_ok,
    'bg_p_1': bg_p_1,
    'autocorr_1': not autocorr_1_ok if autocorr_1_ok is not None else None,
    'bg_p_4': bg_p_4,
    'autocorr_4': not autocorr_4_ok if autocorr_4_ok is not None else None
}

# ============================================================
# 6.4. RESIDUAL DIAGNOSTIC PLOTS
# ============================================================

print("\n" + "="*60)
print("6.4. RESIDUAL DIAGNOSTIC PLOTS")
print("="*60)

from scipy.stats import norm

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'Residual Diagnostics — {best_model_name}\n(training set, n={n_train})',
             fontsize=14, fontweight='bold')

# Plot 1: Residual histogram with normal distribution curve
ax = axes[0, 0]
ax.hist(residuals_train, bins=20, edgecolor='black', alpha=0.7, density=True, color='steelblue')
x_range = np.linspace(residuals_train.min(), residuals_train.max(), 100)
ax.plot(x_range, norm.pdf(x_range, residuals_train.mean(), residuals_train.std()),
        'r-', linewidth=2, label='Normal distribution')
ax.axvline(x=0, color='red', linestyle='--', linewidth=1, alpha=0.5)
ax.set_xlabel('Residuals (bln RUB)')
ax.set_ylabel('Density')
ax.set_title('Residual Distribution')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Plot 2: Q-Q plot
ax = axes[0, 1]
from scipy import stats as sp_stats
sp_stats.probplot(residuals_train, dist="norm", plot=ax)
ax.set_title('Q-Q Plot (normality check)')
ax.grid(True, alpha=0.3)

# Plot 3: Residuals vs fitted values
ax = axes[1, 0]
ax.scatter(y_train_pred_best, residuals_train, alpha=0.6, color='steelblue', edgecolors='white')
ax.axhline(y=0, color='red', linestyle='--', linewidth=1.5)
ax.set_xlabel('Fitted values (bln RUB)')
ax.set_ylabel('Residuals (bln RUB)')
ax.set_title('Residuals vs Fitted\n(homoscedasticity check)')
ax.grid(True, alpha=0.3)

# Plot 4: Autocorrelation function (ACF)
ax = axes[1, 1]
plot_acf(residuals_train, lags=min(20, n_train//4), ax=ax, title='')
ax.set_title('Autocorrelation Function of Residuals (ACF)')
ax.set_xlabel('Lag')
ax.set_ylabel('Autocorrelation')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('01_diagnostics_best_model.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Diagnostic plots saved to '01_diagnostics_best_model.png'")

# ============================================================
# 6.5. DIAGNOSTIC COMPARISON: DW vs BREUSCH-GODFREY
# ============================================================

print("\n" + "="*60)
print("6.5. COMPARISON: DW vs BREUSCH-GODFREY TEST")
print("="*60)

# Calculate DW for comparison (though invalid with lagged variables)
dw_stat = durbin_watson(residuals_train)

print(f"\n📊 Autocorrelation test comparison:")
print(f"   Durbin-Watson test (DW):")
print(f"     DW = {dw_stat:.3f}")
print(f"     {'✅ No autocorrelation (1.5 < DW < 2.5)' if 1.5 < dw_stat < 2.5 else '⚠️ Autocorrelation'}")
print(f"   Breusch-Godfrey test (lag 1):")
print(f"     p = {bg_p_1:.4f}")
print(f"     {'✅ No autocorrelation' if autocorr_1_ok else '⚠️ Autocorrelation'}")
print(f"\n📌 Why Breusch-Godfrey is preferred:")
print(f"   1. DW is biased with lagged dependent variables")
print(f"   2. BG tests autocorrelation at multiple orders")
print(f"   3. BG has higher power for small samples")
print(f"   → Further analysis relies on Breusch-Godfrey test")

# ============================================================
# 6.6. RIDGE REGRESSION COEFFICIENTS (Interpretability)
# ============================================================

print("\n" + "="*60)
print("6.6. RIDGE REGRESSION COEFFICIENTS")
print("="*60)

ridge_model = results['Ridge Regression']['model']
feature_names = X.columns

# Get coefficients (on scaled features)
coef_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': ridge_model.coef_
}).sort_values('coefficient', ascending=False)

print("\n📊 Ridge regression coefficients (scaled features):")
print(coef_df.to_string(index=False))

print("\n📌 Interpretation:")
print("   Positive coefficient → DEPOS INCREASES when factor grows")
print("   Negative coefficient → DEPOS DECREASES when factor grows")

print("\n🔺 TOP-5 DRIVERS (increase deposits):")
print(coef_df.head(5).to_string(index=False))

print("\n🔻 TOP-5 DRAGGERS (decrease deposits):")
print(coef_df.tail(5).to_string(index=False))

# ============================================================
# 6.7. OPTIMAL ALPHA TUNING FOR RIDGE REGRESSION
# ============================================================

print("\n" + "="*60)
print("6.7. OPTIMAL ALPHA TUNING FOR RIDGE REGRESSION")
print("="*60)

from sklearn.linear_model import RidgeCV

# Range of alpha values
alphas = np.logspace(-3, 3, 50)  # from 0.001 to 1000

# RidgeCV with 5-fold cross-validation
ridge_cv = RidgeCV(alphas=alphas, scoring='neg_mean_squared_error', cv=5)
ridge_cv.fit(X_train_scaled, y_train)

best_alpha = ridge_cv.alpha_
print(f"\n✅ Best alpha from cross-validation: {best_alpha:.4f}")

# Train model with best alpha
ridge_optimal = Ridge(alpha=best_alpha)
ridge_optimal.fit(X_train_scaled, y_train)
y_pred_optimal = ridge_optimal.predict(X_test_scaled)

# Compare with original Ridge (alpha=1.0)
r2_original = results['Ridge Regression']['R²']
r2_optimal = r2_score(y_test, y_pred_optimal)
rmse_original = results['Ridge Regression']['RMSE']
rmse_optimal = np.sqrt(mean_squared_error(y_test, y_pred_optimal))

print(f"\n📊 Comparison:")
print(f"   Original Ridge (alpha=1.0):  R² = {r2_original:.4f}, RMSE = {rmse_original:.2f}")
print(f"   Optimal Ridge (alpha={best_alpha:.4f}): R² = {r2_optimal:.4f}, RMSE = {rmse_optimal:.2f}")

# Update best model if optimal is better
if r2_optimal > r2_original:
    print("\n✅ Optimal Ridge outperforms original! Updating best model...")
    results['Ridge Regression (Optimal)'] = {
        'R²': r2_optimal,
        'MAE': mean_absolute_error(y_test, y_pred_optimal),
        'RMSE': rmse_optimal,
        'MAPE': f"{np.mean(np.abs((y_test - y_pred_optimal) / y_test)) * 100:.2f}%",
        'predictions': y_pred_optimal,
        'model': ridge_optimal
    }
    best_model_name = 'Ridge Regression (Optimal)'
    best_pred = y_pred_optimal
else:
    print("\nℹ️ Original Ridge (alpha=1.0) is already optimal.")

# ============================================================
# 6.8. FEATURE SELECTION FOR LINEAR REGRESSION (p-values)
# ============================================================

print("\n" + "="*60)
print("6.8. FEATURE SELECTION FOR LINEAR REGRESSION")
print("="*60)

# Add constant for statsmodels
X_train_const = sm.add_constant(X_train)
X_test_const = sm.add_constant(X_test)

# OLS estimation with statsmodels
ols_full = sm.OLS(y_train, X_train_const).fit()

print("\n📊 Full linear regression summary (p-values):")
print("="*60)
print(ols_full.summary())

# Extract p-values
p_values = ols_full.pvalues
significant_features = p_values[p_values < 0.05].index.tolist()

# Remove 'const' from list
significant_features = [f for f in significant_features if f != 'const']

print(f"\n📊 Features with p-value < 0.05 (statistically significant):")
print(f"   {len(significant_features)} features: {significant_features}")

# Build reduced model with only significant features
if len(significant_features) > 0:
    X_train_reduced = X_train[significant_features]
    X_test_reduced = X_test[significant_features]

    # Train reduced linear regression
    lr_reduced = LinearRegression()
    lr_reduced.fit(X_train_reduced, y_train)
    y_pred_reduced = lr_reduced.predict(X_test_reduced)

    # Metrics
    r2_reduced = r2_score(y_test, y_pred_reduced)
    rmse_reduced = np.sqrt(mean_squared_error(y_test, y_pred_reduced))
    mae_reduced = mean_absolute_error(y_test, y_pred_reduced)

    print(f"\n📊 Reduced linear regression (significant features only):")
    print(f"   R² = {r2_reduced:.4f}")
    print(f"   RMSE = {rmse_reduced:.2f} bln RUB")
    print(f"   MAE = {mae_reduced:.2f} bln RUB")

    # Compare with full linear regression
    r2_original_lr = results['Linear Regression']['R²']
    rmse_original_lr = results['Linear Regression']['RMSE']

    print(f"\n📊 Comparison with full linear regression:")
    print(f"   Full model:   R² = {r2_original_lr:.4f}, RMSE = {rmse_original_lr:.2f}")
    print(f"   Reduced model: R² = {r2_reduced:.4f}, RMSE = {rmse_reduced:.2f}")

    if r2_reduced > r2_original_lr:
        print("   ✅ Reduced model is better (noise removed)")
    else:
        print("   ℹ️ Full model is better (all features contribute)")

    # Print reduced model coefficients
    coef_reduced = pd.DataFrame({
        'feature': significant_features,
        'coefficient': lr_reduced.coef_
    }).sort_values('coefficient', ascending=False)

    print("\n📊 Reduced model coefficients:")
    print(coef_reduced.to_string(index=False))

else:
    print("\n⚠️ No features with p-value < 0.05. All features are insignificant.")

print("\n✅ Model diagnostics completed")

# ============================================================
# 7. FORECAST VISUALIZATION
# ============================================================

print("\n" + "="*60)
print("7. FORECAST VISUALIZATION")
print("="*60)

# 7.1. Historical data vs forecast plot
plt.figure(figsize=(14, 7))

# Historical data (all)
plt.plot(df.index, df['DEPOS'], label='Actual data', color='#1f77b4', linewidth=2.5)

# Best model forecast on test period
best_pred = results[best_model_name]['predictions']
test_dates = X.index[train_size:]

plt.plot(test_dates, best_pred, label=f'Forecast ({best_model_name})',
         color='#ff7f0e', linestyle='--', linewidth=2.5)

# Confidence interval (±2 RMSE)
rmse_best = results[best_model_name]['RMSE']
plt.fill_between(test_dates,
                 best_pred - 2*rmse_best,
                 best_pred + 2*rmse_best,
                 alpha=0.25, color='#ff7f0e', label='95% confidence interval')

plt.title('Household Deposit Volume Forecast in Russia\n' +
          f'Best Model: {best_model_name} (R² = {results_df.loc[best_model_name, "R²"]:.4f})',
          fontsize=14)
plt.xlabel('Date')
plt.ylabel('Deposit Volume, bln RUB')
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('forecast_plot.png', dpi=300, bbox_inches='tight')
plt.show()

# 7.2. Actual vs Predicted (scatter plot)
plt.figure(figsize=(8, 8))

all_y = pd.concat([y_train, y_test])
all_pred = pd.concat([
    pd.Series(results[best_model_name]['model'].predict(X_train_scaled if best_model_name == 'Ridge Regression' else X_train),
              index=y_train.index),
    pd.Series(best_pred, index=y_test.index)
])

plt.scatter(all_y, all_pred, alpha=0.6)
plt.plot([all_y.min(), all_y.max()], [all_y.min(), all_y.max()],
         'r--', linewidth=2, label='Ideal line')
plt.xlabel('Actual values')
plt.ylabel('Predicted values')
plt.title(f'Actual vs Predicted ({best_model_name})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=300, bbox_inches='tight')
plt.show()

# 7.3. Feature importance (Random Forest only)
if 'Random Forest' in results:
    rf_model = results['Random Forest']['model']
    importance = pd.DataFrame({
        'feature': X.columns,
        'importance': rf_model.feature_importances_
    }).sort_values('importance', ascending=False)

    plt.figure(figsize=(10, 8))
    plt.barh(importance['feature'], importance['importance'], color='steelblue')
    plt.xlabel('Importance')
    plt.title('Feature Importance (Random Forest)')
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()

    print("\n📊 TOP-5 MOST IMPORTANT FEATURES:")
    print(importance.head(5))

# ============================================================
# 8. EXPORT RESULTS
# ============================================================

print("\n" + "="*60)
print("8. EXPORT RESULTS")
print("="*60)

# Create DataFrame with forecast results
forecast_df = pd.DataFrame({
    'Date': test_dates,
    'Actual': y_test,
    f'Predicted_{best_model_name}': best_pred,
    'Lower_95%_CI': best_pred - 2*rmse_best,
    'Upper_95%_CI': best_pred + 2*rmse_best
})

# Save to CSV
forecast_df.to_csv('forecast_results.csv', index=False)
print("✅ Forecast results saved to 'forecast_results.csv'")

# Save metrics
results_df.to_csv('model_metrics.csv')
print("✅ Model metrics saved to 'model_metrics.csv'")

# ============================================================
# 9. FINAL SUMMARY
# ============================================================

print("\n" + "="*60)
print("9. FINAL SUMMARY")
print("="*60)

print(f"""
📌 KEY RESULTS OF BLOCK 1:

1. BEST MODEL: {best_model_name}
   ┌─────────────────────────────────────────────────────────┐
   │ TRAINING set (n={n_train}):                            │
   │   R²_train = {r2_train_best:.4f}                       │
   │   R²_adj_train = {r2_adj_train_best:.4f}               │
   │   AIC = {aic_best:.2f}                                 │
   │   BIC = {bic_best:.2f}                                 │
   ├─────────────────────────────────────────────────────────┤
   │ TEST set (n={len(y_test)}):                            │
   │   R²_test = {r2_test_best:.4f}                         │
   │   RMSE = {rmse_test_best:.2f} bln RUB                 │
   │   MAE = {mae_test_best:.2f} bln RUB                   │
   └─────────────────────────────────────────────────────────┘

2. RESIDUAL DIAGNOSTICS (training set):
   - Normality: {'✅' if normality_ok else '⚠️ Deviation' if normality_ok is not None else 'N/A'}
   - Homoscedasticity: {'✅' if homoscedasticity_ok else '⚠️ Heteroscedasticity' if homoscedasticity_ok is not None else 'N/A'}
   - Autocorrelation (lag 1): {'✅ None' if autocorr_1_ok else '⚠️ Present' if autocorr_1_ok is not None else 'N/A'}
   - Autocorrelation (lag 4): {'✅ None' if autocorr_4_ok else '⚠️ Present' if autocorr_4_ok is not None else 'N/A'}

3. RIDGE REGRESSION COEFFICIENTS (TOP DRIVERS):
""")

# Get coefficients from best model (if Ridge)
if 'Ridge' in best_model_name:
    best_coef_df = pd.DataFrame({
        'feature': feature_names,
        'coefficient': best_model.coef_
    }).sort_values('coefficient', ascending=False)

    print("   🔺 TOP-3 DRIVERS (increase deposits):")
    for _, row in best_coef_df.head(3).iterrows():
        print(f"      - {row['feature']}: {row['coefficient']:.4f}")

    print("   🔻 TOP-3 DRAGGERS (decrease deposits):")
    negative_coefs = best_coef_df[best_coef_df['coefficient'] < 0].sort_values('coefficient')
    for _, row in negative_coefs.head(3).iterrows():
        print(f"      - {row['feature']}: {row['coefficient']:.4f}")

print(f"""
4. KEY FINDINGS:
   - Ridge regression with alpha=1.0 performed best on test data
   - Multicollinearity (VIF > 100) successfully handled by regularization
   - Lag features (1, 3, 6, 12 months) are critical for forecasting
   - R²_train - R²_test gap = {r2_diff:.4f} {'(possible overfitting)' if r2_diff > 0.1 else '(good generalization)'}
   - Autocorrelation {'at lag 4 requires attention' if autocorr_4_ok is not None and not autocorr_4_ok else 'at lag 1 requires attention' if autocorr_1_ok is not None and not autocorr_1_ok else 'not detected'}

5. NEXT STEPS:
   - [x] Baseline model built and diagnosed
   - [ ] Feature Engineering: add seasonal and macroeconomic features (Block 4)
   - [ ] Compare with full model across all metrics (R²_train, AIC, BIC)
   - [ ] Address residual autocorrelation (if needed)
   - [ ] Test SARIMA/Prophet for comparison

6. SAVED FILES:
   - correlation_matrix.png — correlation matrix
   - time_series_all.png — time series of all indicators
   - scatter_plots.png — scatter plots
   - depos_distribution.png — target variable distribution
   - forecast_plot.png — forecast plot
   - actual_vs_predicted.png — actual vs predicted
   - feature_importance.png — feature importance (Random Forest)
   - 01_diagnostics_best_model.png — residual diagnostics
   - forecast_results.csv — forecast results
   - model_metrics.csv — model metrics
""")

print("✅ BLOCK 1 SUCCESSFULLY COMPLETED")